# SparkOCR-VLM — Databricks Free Edition

Reads PDFs from a Volume, runs OCR via OpenRouter, writes results to **Unity Catalog** `workspace.default.ocr_silver`.

In [ ]:
# Must touch spark before %pip install on Databricks serverless
print('Spark version:', spark.version)

In [ ]:
%pip install -q /Volumes/workspace/default/sparkocr_demo/sparkocr_vlm-0.1.0-py3-none-any.whl
dbutils.library.restartPython()

In [ ]:
import os
try:
    api_key = dbutils.widgets.get('OPENROUTER_API_KEY')
except Exception:
    api_key = os.environ.get('OPENROUTER_API_KEY', '')
os.environ['OPENROUTER_API_KEY'] = api_key
print('API key loaded:', 'yes' if api_key else 'MISSING')

In [ ]:
INPUT = '/Volumes/workspace/default/sparkocr_demo/input/'
UC_TABLE = 'workspace.default.ocr_silver'

files = dbutils.fs.ls(INPUT)
print(f'PDFs in volume: {len(files)}')
for f in files:
    print(f'  {f.name}  {f.size/1024:.1f} KB')

In [ ]:
from sparkocr_vlm import OCRPipeline

# No output_path — we write to UC ourselves below
pipeline = OCRPipeline(
    backend='openrouter',
    model='nvidia/nemotron-nano-12b-v2-vl:free',
    input_path=INPUT,
    batch_size=1,
    max_cost_usd=0.50,
)

print('Running OCR pipeline...')
silver = pipeline.run(spark)
print(f'Pages processed: {silver.count()}')

In [ ]:
# Write to Unity Catalog Delta table
(
    silver
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(UC_TABLE)
)
print(f'Written to UC table: {UC_TABLE}')

In [ ]:
display(spark.table(UC_TABLE).select(
    'filename', 'page_num', 'markdown', 'prompt_tokens', 'completion_tokens', 'cost_usd', 'error'
))

In [ ]:
%%sql
SELECT
  filename,
  page_num,
  LEFT(markdown, 300) AS markdown_preview,
  cost_usd,
  error
FROM workspace.default.ocr_silver
ORDER BY filename, page_num

In [ ]:
import json

rows = spark.table(UC_TABLE).select('filename','page_num','markdown','cost_usd','error').collect()
total_cost = sum(r['cost_usd'] or 0 for r in rows)
errors = [f"{r['filename']} p{r['page_num']}" for r in rows if r['error']]

summary = {
    'uc_table': UC_TABLE,
    'total_pages': len(rows),
    'total_cost_usd': round(total_cost, 6),
    'errors': errors,
    'pages': [
        {'file': r['filename'], 'page': r['page_num'],
         'markdown': (r['markdown'] or '')[:500], 'cost_usd': r['cost_usd']}
        for r in rows
    ],
}

print(json.dumps(summary, indent=2))
dbutils.notebook.exit(json.dumps(summary))